In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

In [2]:
rng = np.random.default_rng(42)

In [3]:
N_COMPANIES          = 8
USERS_PER_COMPANY    = (40, 120)      # random range per company
SIM_DAYS             = 60
EVENTS_PER_USER      = (15, 70)       # random range per user
RISKY_USER_FRACTION  = 0.50           # fraction of users who carry SOME risk pattern
LABEL_NOISE          = 0.03           # % of labels randomly flipped (real-world messiness)
BENIGN_OUTLIER_RATE  = 0.04 

In [4]:
COUNTRIES  = ['India','USA','Germany','UK','Russia','China','Brazil','Singapore','Nigeria','France']
RESOURCES  = ['Customer_DB','Payroll_DB','Source_Code','Email_Server','Finance_Reports',
              'HR_Records','Cloud_Storage','CRM','Billing_System','Internal_Wiki']
SENSITIVE_RESOURCES = {'Payroll_DB','Finance_Reports','HR_Records','Source_Code'}
THREAT_TYPES = ['insider_threat','data_exfiltration','credential_theft',
                'privilege_abuse','lateral_movement','account_compromise']
 
START_DATE = datetime(2025, 1, 1)

In [5]:
def make_ip(external=False):
    if external:
        return f"{rng.integers(1,223)}.{rng.integers(0,255)}.{rng.integers(0,255)}.{rng.integers(1,255)}"
    return f"192.168.{rng.integers(0,60)}.{rng.integers(1,254)}"

In [6]:
def clip(v, lo, hi):
    return max(lo, min(hi, v))

In [7]:
rows = []
company_ids = [f"T{str(i).zfill(3)}" for i in range(1, N_COMPANIES + 1)]

In [14]:
user_counter = 0
for company_id in company_ids:
    home_country = rng.choice(COUNTRIES)
    n_users = rng.integers(*USERS_PER_COMPANY)
 
    for _ in range(n_users):
        user_counter += 1
        user_id = f"U{str(user_counter).zfill(5)}"
 
        is_risky_user = rng.random() < RISKY_USER_FRACTION
        threat_type = rng.choice(THREAT_TYPES) if is_risky_user else None
 
        # per-user baseline "behavioral fingerprint"
        baseline_login_hour = rng.normal(13, 3)          # most people work daytime
        baseline_hour_std   = rng.uniform(1.2, 2.5)
        baseline_session    = rng.normal(280, 70)         # seconds/minutes unit agnostic
        baseline_transfer   = np.exp(rng.normal(5.2, 0.5))  # lognormal-ish, MB
        fav_resources       = rng.choice(RESOURCES, size=rng.integers(2, 4), replace=False)
        base_failed_lambda  = 0.15
 
        n_events = rng.integers(*EVENTS_PER_USER)
 
        # if risky, escalation starts at a random point and ramps up (gradual insider drift)
        escalation_start = rng.integers(int(SIM_DAYS * 0.05), int(SIM_DAYS * 0.4)) if is_risky_user else None
 
        for e in range(n_events):
            day_offset = rng.integers(0, SIM_DAYS)
            ts = START_DATE + timedelta(days=int(day_offset),
                                         seconds=int(rng.integers(0, 86400)))
 
            # ---- decide if THIS event is drawn from the anomalous process ----
            anomalous = False
            severity = 0.0  # 0-1, controls how far the shift goes (creates overlap)
 
            if is_risky_user and day_offset >= escalation_start:
                days_into_escalation = day_offset - escalation_start
                ramp = clip(days_into_escalation / max(1, (SIM_DAYS - escalation_start)), 0, 1)
                p_anomalous = 0.30 + 0.55 * ramp   # ramps from 30% to ~85% likelihood
                if rng.random() < p_anomalous:
                    anomalous = True
                    severity = clip(rng.beta(2, 2) * (0.4 + 0.6 * ramp), 0, 1)
            elif (not is_risky_user) and rng.random() < BENIGN_OUTLIER_RATE:
                # innocent spike: quarter-end reporting, one-off travel, big legit download, etc.
                anomalous = True
                severity = rng.uniform(0.15, 0.45)  # capped lower severity than real threats
 
            # ---- draw features, with overlap controlled by `severity` ----
            login_hour = rng.normal(baseline_login_hour, baseline_hour_std)
            if anomalous:
                # off-hours shift, but noisy/partial, not absolute
                login_hour += rng.normal(0, 4) * severity * rng.choice([-1, 1])
            login_hour = int(clip(round(login_hour), 0, 23))
 
            session_duration = max(10, rng.normal(baseline_session, 60) *
                                    (1 + rng.normal(0, 0.3) * severity if anomalous else 1))
 
            failed_logins = rng.poisson(base_failed_lambda + (severity * rng.uniform(1, 6) if anomalous else 0))
 
            transfer = baseline_transfer * np.exp(rng.normal(0, 0.4))
            if anomalous:
                transfer *= (1 + rng.gamma(2, 2) * severity)  # heavy-tailed multiplier, overlapping
            data_transfer_amount = round(max(1, transfer), 1)
 
            files_accessed = rng.poisson(8 + (severity * rng.uniform(20, 80) if anomalous else 0))
            emails_sent = rng.poisson(4 + (severity * rng.uniform(5, 15) if anomalous else 0))
            db_queries = rng.poisson(6 + (severity * rng.uniform(15, 60) if anomalous else 0))
            usb_usage = rng.poisson(0.05 + (severity * rng.uniform(0.5, 3) if anomalous else 0))
            vpn_usage = int(rng.random() < (0.15 + (0.5 * severity if anomalous else 0)))
 
            # resource access: anomalous events more likely to hit sensitive/unfamiliar resources
            if anomalous and rng.random() < (0.3 + 0.5 * severity):
                resource = rng.choice(list(SENSITIVE_RESOURCES))
            else:
                resource = rng.choice(fav_resources)
 
            # geography / ip: account_compromise & lateral_movement more likely foreign IP
            foreign_bias = 0.05
            if anomalous and threat_type in ('account_compromise', 'credential_theft', 'lateral_movement'):
                foreign_bias = 0.15 + 0.6 * severity
            elif anomalous:
                foreign_bias = 0.1 + 0.2 * severity
            is_foreign = rng.random() < foreign_bias
            country = rng.choice(COUNTRIES) if is_foreign else home_country
            ip_address = make_ip(external=is_foreign)
 
            is_weekend = ts.weekday() >= 5
 
            rows.append({
                "company_id": company_id,
                "user_id": user_id,
                "login_time": ts.strftime("%Y-%m-%d %H:%M:%S"),
                "login_hour": login_hour,
                "is_weekend": int(is_weekend),
                "session_duration": round(session_duration, 1),
                "failed_login_count": int(failed_logins),
                "data_transfer_amount": data_transfer_amount,
                "files_accessed": int(files_accessed),
                "emails_sent": int(emails_sent),
                "database_queries": int(db_queries),
                "usb_usage": int(usb_usage),
                "vpn_usage": vpn_usage,
                "resource_accessed": resource,
                "country": country,
                "ip_address": ip_address,
                "_anomalous_process": int(anomalous),     # latent ground truth, pre-noise
                "_threat_type": threat_type if anomalous else None,
            })
    
    
df = pd.DataFrame(rows)
 
# ---- final label = latent process flag + small realistic label noise ----
flip_mask = rng.random(len(df)) < LABEL_NOISE
df["risk_label"] = df["_anomalous_process"]
df.loc[flip_mask, "risk_label"] = 1 - df.loc[flip_mask, "risk_label"]
 
# threat_category only meaningful where risk_label==1 (keep it for the explainability layer later)
df["threat_category"] = np.where(df["risk_label"] == 1,
                                  df["_threat_type"].fillna("unclassified_anomaly"),
                                  "none")
 
df = df.drop(columns=["_anomalous_process", "_threat_type"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle rows
 
print("Total rows:", len(df))
print("risk_label distribution:\n", df["risk_label"].value_counts(normalize=True))
print("\nthreat_category breakdown:\n", df["threat_category"].value_counts())
 
df.to_csv("C:/Users/admin/Downloads/SentinelAI/datasets/raw/saas_features_v2.csv", index=False)
print("Saved to saas_features_v2.csv")

Total rows: 111447
risk_label distribution:
 0    0.742757
1    0.257243
Name: risk_label, dtype: float64

threat_category breakdown:
 none                    82778
unclassified_anomaly     4703
credential_theft         4458
account_compromise       4257
insider_threat           4213
lateral_movement         3828
data_exfiltration        3611
privilege_abuse          3599
Name: threat_category, dtype: int64
Saved to saas_features_v2.csv
